#### Importing Libraries

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)

#### Loading the Dataset

In [3]:
data_path = Path("../data/raw/Online Retail.xlsx")

df = pd.read_excel(data_path, engine="openpyxl")

print(f"Dataset Shape: {df.shape}")
print(f"Date Range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

display(df.head())

Dataset Shape: (541909, 8)
Date Range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


#### Saving the Raw Dataset in Parquet Format

In [6]:
parquet_path = Path("../data/processed/online_retail_raw.parquet")
parquet_path.parent.mkdir(parents=True, exist_ok=True)

text_columns = ["InvoiceNo", "StockCode", "Description", "Country"]

df[text_columns] = df[text_columns].astype("string")

df.to_parquet(parquet_path, engine="pyarrow", index=False)

print(f"Parquet file saved to: {parquet_path}")
print(f"File size: {parquet_path.stat().st_size / (1024 ** 2):.2f} MB")

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Data Types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing Values:")
display(df.isna().sum().to_frame("Missing Values"))

Parquet file saved to: ..\data\processed\online_retail_raw.parquet
File size: 3.42 MB

Dataset Shape:
(541909, 8)

Column Data Types:


,Data Type
InvoiceNo,string
StockCode,string
Description,string
Quantity,int64
InvoiceDate,datetime64[us]
UnitPrice,float64
CustomerID,float64
Country,string



Missing Values:


,Missing Values
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


#### Creating Transaction-Level Revenue Variables

In [7]:
eda_df = df.copy()

eda_df["Transaction_Date"] = eda_df["InvoiceDate"].dt.normalize()
eda_df["Line_Amount"] = eda_df["Quantity"] * eda_df["UnitPrice"]
eda_df["Is_Cancellation"] = eda_df["InvoiceNo"].str.startswith("C")

eda_df["Transaction_Type"] = np.select(
    [
        eda_df["Is_Cancellation"],
        eda_df["Quantity"] < 0,
        eda_df["UnitPrice"] < 0,
        eda_df["UnitPrice"] == 0
    ],
    [
        "Cancellation",
        "Other Negative Quantity",
        "Negative Unit Price",
        "Zero Unit Price"
    ],
    default="Positive Quantity and Price"
)

display(
    eda_df[
        [
            "InvoiceNo",
            "StockCode",
            "Transaction_Date",
            "Quantity",
            "UnitPrice",
            "Line_Amount",
            "Is_Cancellation",
            "Transaction_Type"
        ]
    ].head(10)
)

print("\nTransaction Type Distribution:")
display(
    eda_df["Transaction_Type"]
    .value_counts()
    .rename_axis("Transaction_Type")
    .reset_index(name="Records")
)

,InvoiceNo,StockCode,Transaction_Date,Quantity,UnitPrice,Line_Amount,Is_Cancellation,Transaction_Type
0,536365,85123A,2010-12-01,6,2.55,15.30,False,Positive Quantity and Price
1,536365,71053,2010-12-01,6,3.39,20.34,False,Positive Quantity and Price
2,536365,84406B,2010-12-01,8,2.75,22.00,False,Positive Quantity and Price
3,536365,84029G,2010-12-01,6,3.39,20.34,False,Positive Quantity and Price
4,536365,84029E,2010-12-01,6,3.39,20.34,False,Positive Quantity and Price
5,536365,22752,2010-12-01,2,7.65,15.30,False,Positive Quantity and Price
6,536365,21730,2010-12-01,6,4.25,25.50,False,Positive Quantity and Price
7,536366,22633,2010-12-01,6,1.85,11.10,False,Positive Quantity and Price
8,536366,22632,2010-12-01,6,1.85,11.10,False,Positive Quantity and Price
9,536367,84879,2010-12-01,32,1.69,54.08,False,Positive Quantity and Price



Transaction Type Distribution:


,Transaction_Type,Records
0,Positive Quantity and Price,530104
1,Cancellation,9288
2,Other Negative Quantity,1336
3,Zero Unit Price,1179
4,Negative Unit Price,2


**Key Observations**
- **Positive-quantity, positive-price transactions account for 530,104 records (97.82%)**, confirming that conventional sales transactions dominate the dataset.
- **Cancellation invoices account for 9,288 records (1.71%)**, highlighting the importance of distinguishing returns and cancellations from completed sales when calculating revenue.
- **An additional 1,336 records contain negative quantities outside cancellation invoices**, while 1,179 records have zero unit prices and only 2 contain negative unit prices. These entries require separate treatment during revenue construction.
- **Transaction classification provides a foundation for distinguishing merchandise sales from financial adjustments**, but transaction type alone cannot identify all non-merchandise entries, such as postage, fees, and discounts.

**EDA Decision**

Retain all transaction categories during exploratory analysis and compare their financial contributions before establishing the final daily merchandise revenue definition.

#### Analyzing Financial Contributions by Transaction Type

In [8]:
transaction_financial_summary = (
    eda_df.groupby("Transaction_Type")
    .agg(
        Transaction_Records=("Line_Amount", "size"),
        Total_Amount=("Line_Amount", "sum"),
        Average_Line_Amount=("Line_Amount", "mean"),
        Median_Line_Amount=("Line_Amount", "median")
    )
    .reset_index()
)

transaction_financial_summary["Record_Percentage"] = (
    transaction_financial_summary["Transaction_Records"]
    / len(eda_df) * 100
)

transaction_financial_summary["Contribution_to_Net_Percentage"] = (
    transaction_financial_summary["Total_Amount"]
    / eda_df["Line_Amount"].sum() * 100
)

transaction_financial_summary = transaction_financial_summary[
    [
        "Transaction_Type",
        "Transaction_Records",
        "Record_Percentage",
        "Total_Amount",
        "Contribution_to_Net_Percentage",
        "Average_Line_Amount",
        "Median_Line_Amount"
    ]
].sort_values("Total_Amount", ascending=False)

display(transaction_financial_summary)

print(
    f"\nPreliminary Net Transaction Amount: "
    f"{eda_df['Line_Amount'].sum():,.2f}"
)

,Transaction_Type,Transaction_Records,Record_Percentage,Total_Amount,Contribution_to_Net_Percentage,Average_Line_Amount,Median_Line_Amount
3,Positive Quantity and Price,530104,97.82,"10,666,684.54",109.43,20.12,9.90
4,Zero Unit Price,1179,0.22,0.00,0.00,0.00,0.00
2,Other Negative Quantity,1336,0.25,0.00,0.00,0.00,-0.00
1,Negative Unit Price,2,0.00,"-22,124.12",-0.23,"-11,062.06","-11,062.06"
0,Cancellation,9288,1.71,"-896,812.49",-9.20,-96.56,-8.50



Preliminary Net Transaction Amount: 9,747,747.93


**Key Observations**
- **Positive-quantity, positive-price transactions generate 10,666,684.54, representing 109.43% of preliminary net transaction value**, demonstrating that positive financial activity exceeds the final net amount before deductions.
- **Cancellation invoices reduce the preliminary net amount by 896,812.49 (9.20%)**, making them the largest negative financial component and an important consideration when defining gross versus net merchandise revenue.
- **Two negative-price adjustments contribute an additional reduction of 22,124.12 (0.23%)**, while zero-price transactions and other negative-quantity records have no direct monetary contribution.
- **The preliminary net transaction amount totals 9,747,747.93**, reflecting positive transactions after cancellations and negative-price adjustments. This figure is not yet a validated merchandise revenue total because it includes non-merchandise financial entries.

**EDA Decision**

Examine non-merchandise stock codes and their financial contributions before selecting the daily revenue measure. Maintain a clear distinction between gross merchandise sales, merchandise returns, and accounting adjustments to support consistent forecasting.

#### Analyzing Non-Merchandise Transaction Contributions

In [9]:
non_merchandise_codes = [
    "POST", "DOT", "C2", "M", "D", "S",
    "BANK CHARGES", "AMAZONFEE", "CRUK", "B"
]

non_merchandise_df = eda_df[
    eda_df["StockCode"].isin(non_merchandise_codes)
].copy()

non_merchandise_summary = (
    non_merchandise_df.groupby("StockCode")
    .agg(
        Transaction_Records=("InvoiceNo", "size"),
        Unique_Invoices=("InvoiceNo", "nunique"),
        Positive_Amount=("Line_Amount", lambda x: x[x > 0].sum()),
        Negative_Amount=("Line_Amount", lambda x: x[x < 0].sum()),
        Net_Amount=("Line_Amount", "sum")
    )
    .sort_values("Net_Amount", ascending=False)
)

display(non_merchandise_summary)

print("\nOverall Non-Merchandise Summary:")
print(f"Transaction Records: {len(non_merchandise_df):,}")
print(f"Positive Amount: {non_merchandise_df.loc[non_merchandise_df['Line_Amount'] > 0, 'Line_Amount'].sum():,.2f}")
print(f"Negative Amount: {non_merchandise_df.loc[non_merchandise_df['Line_Amount'] < 0, 'Line_Amount'].sum():,.2f}")
print(f"Net Amount: {non_merchandise_df['Line_Amount'].sum():,.2f}")

print("\nShare of Preliminary Net Transaction Amount:")
print(
    f"{non_merchandise_df['Line_Amount'].sum() / eda_df['Line_Amount'].sum() * 100:.2f}%"
)

,Transaction_Records,Unique_Invoices,Positive_Amount,Negative_Amount,Net_Amount
StockCode,,,,,
DOT,710,710,"206,248.77",-3.29,"206,245.48"
POST,1256,1254,"78,101.88","-11,871.24","66,230.64"
C2,144,144,"7,051.00",-65.00,"6,986.00"
S,63,19,63.05,"-3,112.44","-3,049.39"
D,77,65,0.00,"-5,696.22","-5,696.22"
BANK CHARGES,37,36,165.00,"-7,340.64","-7,175.64"
CRUK,16,16,0.00,"-7,933.43","-7,933.43"
B,3,3,"11,062.06","-22,124.12","-11,062.06"
M,571,517,"78,110.27","-146,784.46","-68,674.19"



Overall Non-Merchandise Summary:
Transaction Records: 2,911
Positive Amount: 394,563.12
Negative Amount: -440,212.43
Net Amount: -45,649.31

Share of Preliminary Net Transaction Amount:
-0.47%


In [10]:
display(non_merchandise_summary)

,Transaction_Records,Unique_Invoices,Positive_Amount,Negative_Amount,Net_Amount
StockCode,,,,,
DOT,710,710,"206,248.77",-3.29,"206,245.48"
POST,1256,1254,"78,101.88","-11,871.24","66,230.64"
C2,144,144,"7,051.00",-65.00,"6,986.00"
S,63,19,63.05,"-3,112.44","-3,049.39"
D,77,65,0.00,"-5,696.22","-5,696.22"
BANK CHARGES,37,36,165.00,"-7,340.64","-7,175.64"
CRUK,16,16,0.00,"-7,933.43","-7,933.43"
B,3,3,"11,062.06","-22,124.12","-11,062.06"
M,571,517,"78,110.27","-146,784.46","-68,674.19"


**Key Observations**
- **The ten identified non-merchandise stock codes account for 2,911 transaction records**, generating 394,563.12 in positive amounts and -440,212.43 in negative amounts, resulting in a net contribution of -45,649.31 (-0.47% of preliminary net transaction value).
- **DOT and POST contribute the largest positive net amounts, at 206,245.48 and 66,230.64, respectively**, demonstrating that postage-related charges can inflate reported merchandise revenue when included in transaction totals.
- **AMAZONFEE generates the largest negative net contribution (-221,520.50)**, followed by manual entries under stock code M (-68,674.19), highlighting the financial significance of administrative and accounting adjustments.
- **Other non-merchandise codes represent discounts, bank charges, commissions, carriage, samples, and bad-debt adjustments**, reinforcing the need to distinguish operational charges from product sales.
- **The relatively small overall net contribution conceals substantial offsetting positive and negative amounts**, meaning that evaluating non-merchandise transactions solely by their net percentage would understate their influence on revenue calculations.

**EDA Decision**

Exclude clearly identifiable non-merchandise charges and accounting adjustments from the eventual merchandise revenue target, subject to validating the transaction classifications. Investigate ambiguous manual entries separately before establishing final filtering rules.

### Univariate Analysis
#### Analyzing the Distribution of Daily Sales Revenue

In [13]:
import plotly.graph_objects as go

non_merchandise_codes = [
    "POST", "DOT", "C2", "M", "D",
    "S", "BANK CHARGES", "AMAZONFEE", "CRUK", "B"
]

merchandise_df = eda_df.loc[
    ~eda_df["StockCode"].isin(non_merchandise_codes)
].copy()

daily_revenue = (
    merchandise_df.groupby("Transaction_Date")["Line_Amount"]
    .sum()
    .sort_index()
)

mean_revenue = daily_revenue.mean()
median_revenue = daily_revenue.median()

fig = go.Figure()

fig.add_trace(
    go.Histogram(
        x=daily_revenue,
        nbinsx=30,
        name="Daily Revenue",
        marker=dict(
            color="#4C78A8",
            line=dict(color="white", width=1)
        ),
        opacity=0.85,
        hovertemplate="Daily Revenue: %{x:,.2f}<br>Number of Days: %{y}<extra></extra>"
    )
)

fig.add_vline(
    x=mean_revenue,
    line=dict(color="#E45756", width=2, dash="dash")
)

fig.add_vline(
    x=median_revenue,
    line=dict(color="#2A9D8F", width=2, dash="dash")
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#E45756", width=2, dash="dash"),
        name=f"Mean: {mean_revenue:,.2f}"
    )
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#2A9D8F", width=2, dash="dash"),
        name=f"Median: {median_revenue:,.2f}"
    )
)

fig.update_layout(
    title=dict(
        text="Distribution of Daily Retail Sales Revenue",
        x=0.5,
        xanchor="center"
    ),
    xaxis_title="Daily Sales Revenue",
    yaxis_title="Number of Days",
    template="plotly_white",
    width=1100,
    height=550,
    bargap=0.05,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    hovermode="closest"
)

fig.show()

display(
    daily_revenue.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame("Daily_Revenue")
)

,Daily_Revenue
count,305.00
mean,"32,109.50"
std,"17,382.67"
min,"3,457.11"
25%,"20,861.90"
50%,"27,871.61"
75%,"42,007.09"
90%,"55,465.13"
95%,"62,649.26"
99%,"81,072.38"


**Key Observations**
- **Daily sales revenue exhibits a positively skewed distribution**, with a mean of 32,109.50 exceeding the median of 27,871.61, indicating that higher-revenue days pull the average upward.
- **The middle 50% of trading days generate between 20,861.90 and 42,007.09 in revenue**, while the standard deviation of 17,382.67 highlights substantial variation in daily sales performance.
- **Daily revenue ranges from 3,457.11 to 110,806.17**, with the 95th and 99th percentiles reaching 62,649.26 and 81,072.38, respectively, revealing a small number of exceptionally high-revenue days.

**Business Insights**
- **Considerable variation in daily revenue highlights the importance of reliable short-term forecasting for inventory planning, staffing, and operational resource allocation.**
- **The right-skewed distribution suggests that forecasting models should be evaluated across both typical and high-revenue periods**, ensuring that strong performance on ordinary trading days does not conceal larger errors during sales peaks.

#### Analyzing Daily Sales Revenue Outliers

In [14]:
import plotly.graph_objects as go

q1 = daily_revenue.quantile(0.25)
q3 = daily_revenue.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

revenue_outliers = daily_revenue[
    (daily_revenue < lower_bound) |
    (daily_revenue > upper_bound)
]

fig = go.Figure()

fig.add_trace(
    go.Box(
        x=daily_revenue.values,
        y=["Daily Revenue"] * len(daily_revenue),
        orientation="h",
        name="Daily Revenue",
        boxpoints="outliers",
        quartilemethod="linear",
        marker=dict(
            color="#D9534F",
            size=7,
            opacity=0.7
        ),
        line=dict(color="#377EB8", width=2),
        fillcolor="rgba(55, 126, 184, 0.35)",
        customdata=daily_revenue.index.strftime("%Y-%m-%d"),
        hovertemplate=(
            "Daily Revenue: %{x:,.2f}"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title=dict(
        text="Distribution and Outliers in Daily Retail Sales Revenue",
        x=0.5,
        xanchor="center"
    ),
    xaxis_title="Daily Sales Revenue",
    yaxis_title="",
    template="plotly_white",
    width=1100,
    height=450,
    showlegend=False,
    font=dict(size=13)
)

fig.show()

outlier_summary = pd.DataFrame({
    "Metric": [
        "Total Trading Days",
        "Lower Outlier Bound",
        "Upper Outlier Bound",
        "Lower Outlier Days",
        "Upper Outlier Days",
        "Total Outlier Days",
        "Outlier Percentage",
        "Minimum Daily Revenue",
        "Maximum Daily Revenue",
        "Average Outlier Revenue"
    ],
    "Value": [
        len(daily_revenue),
        lower_bound,
        upper_bound,
        (daily_revenue < lower_bound).sum(),
        (daily_revenue > upper_bound).sum(),
        len(revenue_outliers),
        len(revenue_outliers) / len(daily_revenue) * 100,
        daily_revenue.min(),
        daily_revenue.max(),
        revenue_outliers.mean() if len(revenue_outliers) else np.nan
    ]
})

display(outlier_summary)

print("\nOutlier Dates and Revenue:")
display(
    revenue_outliers.sort_values(ascending=False)
    .rename("Daily_Revenue")
    .to_frame()
)

,Metric,Value
0,Total Trading Days,305.00
1,Lower Outlier Bound,"-10,855.88"
2,Upper Outlier Bound,"73,724.87"
3,Lower Outlier Days,0.00
4,Upper Outlier Days,9.00
5,Total Outlier Days,9.00
6,Outlier Percentage,2.95
7,Minimum Daily Revenue,"3,457.11"
8,Maximum Daily Revenue,"110,806.17"
9,Average Outlier Revenue,"84,805.36"



Outlier Dates and Revenue:


,Daily_Revenue
Transaction_Date,
2011-11-14,"110,806.17"
2011-09-20,"108,335.17"
2011-11-07,"82,628.60"
2010-12-07,"81,131.02"
2011-12-05,"79,665.11"
2011-12-08,"76,617.42"
2011-11-23,"75,496.41"
2011-10-05,"74,470.06"
2011-09-15,"74,098.28"


**Key Observations**
- **Nine trading days (2.95%) are identified as upper revenue outliers**, exceeding the IQR-based threshold of 73,724.87, while no lower outliers are detected.
- **Outlier days generate an average revenue of 84,805.36**, substantially exceeding the overall daily average of 32,109.50, indicating exceptionally strong sales activity on these dates.
- **November 14, 2011 records the highest daily revenue (110,806.17)**, followed by September 20, 2011 (108,335.17). Eight of the nine outlier dates occur between September and December 2011, suggesting a concentration of high-revenue activity toward the end of the observation period.

**Business Insights**
- **High-revenue outliers highlight periods of exceptional retail activity that may require additional inventory, staffing, and fulfillment capacity.**
- **The concentration of outliers in later months suggests that forecasting models should account for changing revenue levels and potential seasonal effects**, rather than treating all unusually high sales days as erroneous observations.

#### Analyzing Daily Transaction Volume

In [15]:
daily_transaction_volume = (
    merchandise_df.groupby("Transaction_Date")
    .size()
    .sort_index()
)

fig = go.Figure()

fig.add_trace(
    go.Histogram(
        x=daily_transaction_volume,
        nbinsx=30,
        name="Transaction Volume",
        marker=dict(
            color="#4C78A8",
            line=dict(color="white", width=1)
        ),
        opacity=0.85,
        hovertemplate=(
            "Daily Transaction Volume: %{x:,}"
            "<br>Number of Days: %{y}"
            "<extra></extra>"
        )
    )
)

fig.add_vline(
    x=daily_transaction_volume.mean(),
    line=dict(color="#E45756", width=2, dash="dash")
)

fig.add_vline(
    x=daily_transaction_volume.median(),
    line=dict(color="#2A9D8F", width=2, dash="dash")
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#E45756", width=2, dash="dash"),
        name=f"Mean: {daily_transaction_volume.mean():,.0f}"
    )
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#2A9D8F", width=2, dash="dash"),
        name=f"Median: {daily_transaction_volume.median():,.0f}"
    )
)

fig.update_layout(
    title=dict(
        text="Distribution of Daily Retail Transaction Volume",
        x=0.5,
        xanchor="center"
    ),
    xaxis_title="Daily Transaction Volume (Invoice Lines)",
    yaxis_title="Number of Days",
    template="plotly_white",
    width=1100,
    height=550,
    bargap=0.05,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig.show()

display(
    daily_transaction_volume.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame("Daily_Transaction_Volume")
)

,Daily_Transaction_Volume
count,305.00
mean,"1,767.21"
std,826.43
min,279.00
25%,"1,222.00"
50%,"1,566.00"
75%,"2,102.00"
90%,"2,937.20"
95%,"3,387.00"
99%,"4,173.52"


**Key Observations**
- **Daily transaction volume averages 1,767 invoice lines, with a median of 1,566**, indicating a positively skewed distribution influenced by days with unusually high transaction activity.
- **The middle 50% of trading days record between 1,222 and 2,102 invoice lines**, while a standard deviation of 826.43 highlights considerable variation in daily transaction activity.
- **Daily transaction volume ranges from 279 to 5,312 invoice lines**, with the 95th and 99th percentiles reaching 3,387 and 4,173.52, respectively, indicating occasional periods of exceptionally high activity.

**Business Insights**
- **Substantial variation in daily transaction volume highlights the importance of anticipating fluctuations in operational workload**, particularly for order processing, inventory handling, and fulfillment planning.
- **High-volume trading days may place additional pressure on retail operations**, making transaction volume a useful variable to investigate when analyzing daily revenue patterns and developing forecasting features.

#### Analyzing Daily Invoice Volume

In [16]:
daily_invoice_volume = (
    merchandise_df.groupby("Transaction_Date")["InvoiceNo"]
    .nunique()
    .sort_index()
)

mean_invoices = daily_invoice_volume.mean()
median_invoices = daily_invoice_volume.median()

fig = go.Figure()

fig.add_trace(
    go.Histogram(
        x=daily_invoice_volume,
        nbinsx=30,
        name="Daily Invoice Volume",
        marker=dict(
            color="#4C78A8",
            line=dict(color="white", width=1)
        ),
        opacity=0.85,
        hovertemplate=(
            "Daily Invoices: %{x:,}"
            "<br>Number of Days: %{y}"
            "<extra></extra>"
        )
    )
)

fig.add_vline(
    x=mean_invoices,
    line=dict(color="#E45756", width=2, dash="dash")
)

fig.add_vline(
    x=median_invoices,
    line=dict(color="#2A9D8F", width=2, dash="dash")
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#E45756", width=2, dash="dash"),
        name=f"Mean: {mean_invoices:,.0f}"
    )
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#2A9D8F", width=2, dash="dash"),
        name=f"Median: {median_invoices:,.0f}"
    )
)

fig.update_layout(
    title=dict(
        text="Distribution of Daily Retail Invoice Volume",
        x=0.5,
        xanchor="center"
    ),
    xaxis_title="Number of Unique Invoices",
    yaxis_title="Number of Days",
    template="plotly_white",
    width=1100,
    height=550,
    bargap=0.05,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig.show()

display(
    daily_invoice_volume.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame("Daily_Invoice_Volume")
)

,Daily_Invoice_Volume
count,305.00
mean,82.95
std,33.51
min,11.00
25%,61.00
50%,80.00
75%,102.00
90%,128.20
95%,147.00
99%,168.96


**Key Observations**
- **Daily invoice volume averages 82.95 unique invoices, with a median of 80**, indicating a slightly positively skewed distribution with occasional higher-activity trading days.
- **The middle 50% of trading days record between 61 and 102 unique invoices**, while a standard deviation of 33.51 reflects considerable variation in daily invoicing activity.
- **Daily invoice volume ranges from 11 to 214 unique invoices**, with the 95th and 99th percentiles reaching 147 and 168.96, respectively, highlighting occasional periods of exceptionally high invoice activity.

**Business Insights**
- **Variations in daily invoice volume highlight changing order-processing and billing workloads**, supporting the need for flexible operational capacity during busier trading periods.
- **Invoice volume provides a complementary measure of retail activity alongside invoice-line volume**, making it valuable for investigating whether daily revenue fluctuations are associated with changes in order frequency or invoice composition.

#### Analyzing Daily Units Sold

In [17]:
daily_units_sold = (
    merchandise_df.groupby("Transaction_Date")["Quantity"]
    .sum()
    .sort_index()
)

mean_units = daily_units_sold.mean()
median_units = daily_units_sold.median()

fig = go.Figure()

fig.add_trace(
    go.Histogram(
        x=daily_units_sold,
        nbinsx=30,
        name="Daily Net Units",
        marker=dict(
            color="#4C78A8",
            line=dict(color="white", width=1)
        ),
        opacity=0.85,
        hovertemplate=(
            "Daily Net Units: %{x:,.0f}"
            "<br>Number of Days: %{y}"
            "<extra></extra>"
        )
    )
)

fig.add_vline(
    x=mean_units,
    line=dict(color="#E45756", width=2, dash="dash")
)

fig.add_vline(
    x=median_units,
    line=dict(color="#2A9D8F", width=2, dash="dash")
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#E45756", width=2, dash="dash"),
        name=f"Mean: {mean_units:,.0f}"
    )
)

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="lines",
        line=dict(color="#2A9D8F", width=2, dash="dash"),
        name=f"Median: {median_units:,.0f}"
    )
)

fig.update_layout(
    title=dict(
        text="Distribution of Daily Net Merchandise Units",
        x=0.5,
        xanchor="center"
    ),
    xaxis_title="Daily Net Merchandise Units",
    yaxis_title="Number of Days",
    template="plotly_white",
    width=1100,
    height=550,
    bargap=0.05,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    )
)

fig.show()

display(
    daily_units_sold.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame("Daily_Net_Units")
)

,Daily_Net_Units
count,305.00
mean,"16,938.51"
std,"8,948.51"
min,"-13,657.00"
25%,"10,708.00"
50%,"15,611.00"
75%,"21,188.00"
90%,"29,661.80"
95%,"33,453.40"
99%,"43,198.52"


**Key Observations**
- **Daily net merchandise quantities average 16,938.51 units, with a median of 15,611**, indicating a positively skewed distribution influenced by higher-volume trading days.
- **The middle 50% of trading days record between 10,708 and 21,188 net units**, while a standard deviation of 8,948.51 highlights substantial variation in daily merchandise movement.
- **Daily net quantities range from -13,657 to 46,140 units**, with the negative minimum indicating that returns or cancellations exceeded positive quantities on at least one trading day. The 95th and 99th percentiles reach 33,453.40 and 43,198.52 units, respectively.

**Business Insights**
- **Considerable fluctuations in merchandise quantities highlight the importance of anticipating changes in product movement**, supporting inventory replenishment, warehouse capacity, and fulfillment planning.
- **Negative net quantities demonstrate how returns and cancellations can distort daily sales activity**, making it important to distinguish net merchandise movement from gross sales when investigating revenue patterns and developing forecasting features.

#### Bivariate Analysis

#### Analyzing Daily Sales Revenue by Day of the Week

In [20]:
weekday_revenue = daily_revenue.rename("Daily_Revenue").reset_index()

weekday_revenue["Day_of_Week"] = (
    weekday_revenue["Transaction_Date"].dt.day_name()
)

weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

fig = px.box(
    weekday_revenue,
    x="Day_of_Week",
    y="Daily_Revenue",
    category_orders={"Day_of_Week": weekday_order},
    points="outliers",
    color="Day_of_Week",
    title="Distribution of Daily Retail Sales Revenue by Day of the Week",
    labels={
        "Day_of_Week": "Day of the Week",
        "Daily_Revenue": "Daily Sales Revenue"
    },
    template="plotly_white"
)

fig.update_layout(
    width=1100,
    height=550,
    title_x=0.5,
    showlegend=False,
    xaxis_title="Day of the Week",
    yaxis_title="Daily Sales Revenue",
    font=dict(size=13)
)

fig.show()

weekday_summary = (
    weekday_revenue.groupby("Day_of_Week")["Daily_Revenue"]
    .agg(
        Trading_Days="count",
        Mean_Revenue="mean",
        Median_Revenue="median",
        Std_Revenue="std",
        Min_Revenue="min",
        Max_Revenue="max"
    )
    .reindex(weekday_order)
)

display(weekday_summary.dropna(how="all"))

,Trading_Days,Mean_Revenue,Median_Revenue,Std_Revenue,Min_Revenue,Max_Revenue
Day_of_Week,,,,,,
Monday,47.00,"34,589.99","27,776.41","18,824.34","16,018.84","110,806.17"
Tuesday,52.00,"37,858.76","33,034.13","18,171.77","14,309.63","108,335.17"
Wednesday,53.00,"32,887.11","27,871.61","15,853.93","4,791.17","75,496.41"
Thursday,53.00,"39,329.51","35,522.79","16,089.95","10,989.24","76,617.42"
Friday,50.00,"31,474.25","26,653.33","13,826.17","12,533.49","62,784.26"
Sunday,50.00,"15,956.39","12,322.94","10,047.12","3,457.11","42,552.66"


**Key Observations**
- **Thursday records the highest average daily revenue (39,329.51), followed by Tuesday (37,858.76)**, while Sunday generates the lowest average revenue (15,956.39), indicating substantial differences in sales performance across trading days.
- **Sunday's median revenue (12,322.94) is considerably lower than the other trading days**, while Thursday records the highest median (35,522.79), suggesting that weekday differences extend beyond occasional high-revenue outliers.
- **Monday exhibits the greatest revenue variability (standard deviation of 18,824.34)** and the highest single-day revenue (110,806.17). No Saturday transactions are recorded, leaving six observed trading days per week.

**Business Insights**
- **Strong weekday differences suggest that day-of-week patterns could be valuable predictors of daily retail revenue**, supporting the inclusion of calendar-based features in forecasting models.
- **Lower Sunday revenue and stronger Tuesday–Thursday performance highlight opportunities to align inventory availability, staffing, and fulfillment capacity with recurring trading-day demand patterns.**

#### Analyzing Daily Sales Revenue by Month

In [21]:
monthly_revenue = daily_revenue.rename("Daily_Revenue").reset_index()

monthly_revenue["Year_Month"] = (
    monthly_revenue["Transaction_Date"].dt.to_period("M").astype(str)
)

monthly_summary = (
    monthly_revenue.groupby("Year_Month")["Daily_Revenue"]
    .agg(
        Trading_Days="count",
        Total_Revenue="sum",
        Mean_Daily_Revenue="mean",
        Median_Daily_Revenue="median",
        Std_Daily_Revenue="std"
    )
    .reset_index()
    .sort_values("Year_Month")
)

fig = px.bar(
    monthly_summary,
    x="Year_Month",
    y="Mean_Daily_Revenue",
    color="Mean_Daily_Revenue",
    color_continuous_scale="Blues",
    title="Average Daily Retail Sales Revenue by Month",
    labels={
        "Year_Month": "Month",
        "Mean_Daily_Revenue": "Average Daily Sales Revenue"
    },
    custom_data=[
        "Trading_Days",
        "Total_Revenue",
        "Median_Daily_Revenue"
    ],
    template="plotly_white"
)

fig.update_traces(
    hovertemplate=(
        "Month: %{x}"
        "<br>Average Daily Revenue: %{y:,.2f}"
        "<br>Trading Days: %{customdata[0]:,.0f}"
        "<br>Total Revenue: %{customdata[1]:,.2f}"
        "<br>Median Daily Revenue: %{customdata[2]:,.2f}"
        "<extra></extra>"
    )
)

fig.update_layout(
    width=1100,
    height=550,
    title_x=0.5,
    xaxis_title="Month",
    yaxis_title="Average Daily Sales Revenue",
    coloraxis_showscale=False,
    xaxis_tickangle=-45,
    font=dict(size=13)
)

fig.show()

display(monthly_summary)

,Year_Month,Trading_Days,Total_Revenue,Mean_Daily_Revenue,Median_Daily_Revenue,Std_Daily_Revenue
0,2010-12,20,"760,403.05","38,020.15","42,691.33","18,675.87"
1,2011-01,24,"580,466.11","24,186.09","22,596.61","12,833.50"
2,2011-02,24,"500,577.00","20,857.38","21,844.86","8,736.75"
3,2011-03,27,"680,590.64","25,207.06","24,296.87","11,840.11"
4,2011-04,21,"482,992.88","22,999.66","23,961.66","7,159.52"
5,2011-05,25,"732,327.51","29,293.10","28,389.52","12,129.38"
6,2011-06,26,"725,116.04","27,889.08","23,179.31","12,739.42"
7,2011-07,26,"677,988.53","26,076.48","25,289.70","11,604.84"
8,2011-08,26,"702,675.64","27,025.99","23,963.67","15,457.62"
9,2011-09,26,"1,013,455.68","38,979.06","35,692.76","18,826.56"


**Key Observations**
- **November 2011 records the highest average daily revenue among complete months (55,105.19)**, followed by October (40,872.86) and September (38,979.06), indicating stronger sales performance toward the end of the year.
- **February 2011 records the lowest average daily revenue (20,857.38)**, while November generates approximately 2.64 times February's daily average, highlighting substantial monthly variation in retail sales activity.
- **Average daily revenue increases from 27,025.99 in August to 55,105.19 in November**, accompanied by November's highest monthly total revenue of 1,432,734.99. December 2010 and December 2011 contain only 20 and 8 observed trading days, respectively, limiting direct comparisons with complete months.

**Business Insights**
- **Stronger revenue performance during September–November suggests a potential seasonal increase in retail demand**, highlighting the importance of incorporating calendar-based patterns into forecasting models.
- **Substantial monthly revenue differences emphasize the need for flexible inventory, staffing, and fulfillment planning**, particularly during periods of increased sales activity.

#### Analyzing the Relationship Between Daily Sales Revenue and Transaction Volume

In [22]:
revenue_volume_df = pd.DataFrame({
    "Daily_Revenue": daily_revenue,
    "Transaction_Volume": daily_transaction_volume
}).dropna()

correlation = revenue_volume_df[
    ["Daily_Revenue", "Transaction_Volume"]
].corr().iloc[0, 1]

fig = px.scatter(
    revenue_volume_df.reset_index(),
    x="Transaction_Volume",
    y="Daily_Revenue",
    trendline="ols",
    opacity=0.7,
    title="Daily Sales Revenue vs. Transaction Volume",
    labels={
        "Transaction_Volume": "Daily Transaction Volume (Invoice Lines)",
        "Daily_Revenue": "Daily Sales Revenue"
    },
    hover_data=["Transaction_Date"],
    template="plotly_white"
)

fig.update_traces(
    marker=dict(size=8, color="#4C78A8"),
    selector=dict(mode="markers")
)

fig.update_layout(
    width=1100,
    height=550,
    title_x=0.5,
    showlegend=False,
    font=dict(size=13)
)

fig.show()

relationship_summary = pd.DataFrame({
    "Metric": [
        "Trading Days",
        "Pearson Correlation",
        "R-Squared",
        "Average Daily Revenue",
        "Average Transaction Volume"
    ],
    "Value": [
        len(revenue_volume_df),
        correlation,
        correlation ** 2,
        revenue_volume_df["Daily_Revenue"].mean(),
        revenue_volume_df["Transaction_Volume"].mean()
    ]
})

display(relationship_summary)

,Metric,Value
0,Trading Days,305.00
1,Pearson Correlation,0.73
2,R-Squared,0.53
3,Average Daily Revenue,"32,109.50"
4,Average Transaction Volume,"1,767.21"


**Key Observations**
- **Daily sales revenue and transaction volume exhibit a strong positive correlation (r = 0.73)**, indicating that trading days with more merchandise invoice lines generally generate higher revenue.
- **The R-squared value of 0.53 indicates that approximately 53% of the variation in daily revenue is explained by a simple linear relationship with transaction volume**, leaving considerable variation associated with other factors.

**Business Insights**
- **Transaction volume is strongly associated with daily revenue performance**, highlighting the importance of retail activity levels when analyzing revenue fluctuations and identifying potential demand drivers.
- **Transaction volume may provide useful forecasting information through historical trends and lagged features**, while same-day transaction counts should not be used unless they are genuinely available at the time of prediction.

#### Analyzing the Relationship Between Daily Sales Revenue and Invoice Volume

In [23]:
revenue_invoice_df = pd.DataFrame({
    "Daily_Revenue": daily_revenue,
    "Invoice_Volume": daily_invoice_volume
}).dropna()

invoice_correlation = revenue_invoice_df[
    ["Daily_Revenue", "Invoice_Volume"]
].corr().iloc[0, 1]

fig = px.scatter(
    revenue_invoice_df.reset_index(),
    x="Invoice_Volume",
    y="Daily_Revenue",
    trendline="ols",
    opacity=0.7,
    title="Daily Sales Revenue vs. Invoice Volume",
    labels={
        "Invoice_Volume": "Daily Unique Invoices",
        "Daily_Revenue": "Daily Sales Revenue"
    },
    hover_data=["Transaction_Date"],
    template="plotly_white"
)

fig.update_traces(
    marker=dict(size=8, color="#4C78A8"),
    selector=dict(mode="markers")
)

fig.update_layout(
    width=1100,
    height=550,
    title_x=0.5,
    showlegend=False,
    font=dict(size=13)
)

fig.show()

invoice_relationship_summary = pd.DataFrame({
    "Metric": [
        "Trading Days",
        "Pearson Correlation",
        "R-Squared",
        "Average Daily Revenue",
        "Average Invoice Volume"
    ],
    "Value": [
        len(revenue_invoice_df),
        invoice_correlation,
        invoice_correlation ** 2,
        revenue_invoice_df["Daily_Revenue"].mean(),
        revenue_invoice_df["Invoice_Volume"].mean()
    ]
})

display(invoice_relationship_summary)

,Metric,Value
0,Trading Days,305.00
1,Pearson Correlation,0.72
2,R-Squared,0.51
3,Average Daily Revenue,"32,109.50"
4,Average Invoice Volume,82.95


**Key Observations**
- **Daily sales revenue and invoice volume exhibit a strong positive correlation (r = 0.72)**, indicating that trading days with more unique invoices generally generate higher revenue.
- **The R-squared value of 0.51 indicates that approximately 51% of daily revenue variation is explained by a simple linear relationship with invoice volume**, slightly below the 53% associated with transaction volume.

**Business Insights**
- **Invoice volume is strongly associated with daily revenue performance**, highlighting the importance of order frequency in understanding retail sales fluctuations.
- **The similar correlations for invoice volume (0.72) and transaction volume (0.73) suggest that both measures capture important aspects of retail activity**, making their historical patterns worth investigating as potential forecasting features.

#### Analyzing the Relationship Between Daily Sales Revenue and Net Merchandise Units

In [24]:
revenue_units_df = pd.DataFrame({
    "Daily_Revenue": daily_revenue,
    "Net_Units": daily_units_sold
}).dropna()

units_correlation = revenue_units_df[
    ["Daily_Revenue", "Net_Units"]
].corr().iloc[0, 1]

fig = px.scatter(
    revenue_units_df.reset_index(),
    x="Net_Units",
    y="Daily_Revenue",
    trendline="ols",
    opacity=0.7,
    title="Daily Sales Revenue vs. Net Merchandise Units",
    labels={
        "Net_Units": "Daily Net Merchandise Units",
        "Daily_Revenue": "Daily Sales Revenue"
    },
    hover_data=["Transaction_Date"],
    template="plotly_white"
)

fig.update_traces(
    marker=dict(size=8, color="#4C78A8"),
    selector=dict(mode="markers")
)

fig.update_layout(
    width=1100,
    height=550,
    title_x=0.5,
    showlegend=False,
    font=dict(size=13)
)

fig.show()

units_relationship_summary = pd.DataFrame({
    "Metric": [
        "Trading Days",
        "Pearson Correlation",
        "R-Squared",
        "Average Daily Revenue",
        "Average Net Merchandise Units",
        "Minimum Net Merchandise Units",
        "Maximum Net Merchandise Units"
    ],
    "Value": [
        len(revenue_units_df),
        units_correlation,
        units_correlation ** 2,
        revenue_units_df["Daily_Revenue"].mean(),
        revenue_units_df["Net_Units"].mean(),
        revenue_units_df["Net_Units"].min(),
        revenue_units_df["Net_Units"].max()
    ]
})

display(units_relationship_summary)

,Metric,Value
0,Trading Days,305.00
1,Pearson Correlation,0.88
2,R-Squared,0.77
3,Average Daily Revenue,"32,109.50"
4,Average Net Merchandise Units,"16,938.51"
5,Minimum Net Merchandise Units,"-13,657.00"
6,Maximum Net Merchandise Units,"46,140.00"


**Key Observations**
- **Daily sales revenue and net merchandise units exhibit a strong positive correlation (r = 0.88)**, indicating that trading days with higher net merchandise quantities generally generate higher revenue.
- **The R-squared value of 0.77 indicates that approximately 77% of daily revenue variation is explained by a simple linear relationship with net merchandise units**, representing a stronger association than transaction volume (r = 0.73) and invoice volume (r = 0.72).

**Business Insights**
- **Net merchandise quantities show the strongest relationship with daily revenue among the three activity measures examined**, highlighting the importance of product movement in understanding retail sales performance.
- **Historical merchandise quantities may provide valuable information for revenue forecasting**, particularly through lagged and rolling features, provided that only information available at the forecast origin is used.

### Multivariate Analysis

#### Analyzing Correlations Among Daily Retail Metrics

In [25]:
daily_metrics = pd.DataFrame({
    "Daily Revenue": daily_revenue,
    "Transaction Volume": daily_transaction_volume,
    "Invoice Volume": daily_invoice_volume,
    "Net Merchandise Units": daily_units_sold
}).dropna()

correlation_matrix = daily_metrics.corr(method="pearson")

fig = px.imshow(
    correlation_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Correlation Heatmap of Daily Retail Metrics",
    labels={
        "x": "Retail Metrics",
        "y": "Retail Metrics",
        "color": "Pearson Correlation"
    },
    template="plotly_white"
)

fig.update_traces(
    hovertemplate=(
        "%{y} vs. %{x}"
        "<br>Pearson Correlation: %{z:.3f}"
        "<extra></extra>"
    )
)

fig.update_layout(
    width=950,
    height=650,
    title_x=0.5,
    font=dict(size=13),
    coloraxis_colorbar=dict(title="Pearson r")
)

fig.show()

display(correlation_matrix.round(3))

,Daily Revenue,Transaction Volume,Invoice Volume,Net Merchandise Units
Daily Revenue,1.00,0.72,0.72,0.88
Transaction Volume,0.72,1.00,0.76,0.67
Invoice Volume,0.72,0.76,1.00,0.70
Net Merchandise Units,0.88,0.67,0.70,1.00


**Key Observations**
- **Daily revenue exhibits the strongest positive correlation with net merchandise units (r = 0.88)**, followed by transaction volume and invoice volume (both r = 0.72), indicating that merchandise quantities have the strongest linear association with revenue.
- **Transaction volume and invoice volume are strongly correlated (r = 0.76)**, suggesting that days with more merchandise invoice lines generally involve more unique invoices.
- **Net merchandise units also correlate positively with invoice volume (r = 0.70) and transaction volume (r = 0.67)**, revealing substantial relationships among the different measures of retail activity.

**Business Insights**
- **Merchandise quantities, transaction volume, and invoice volume collectively reflect interconnected aspects of retail activity**, providing useful context for understanding fluctuations in daily sales revenue.
- **Strong correlations among retail activity measures suggest potential redundancy in forecasting features**, highlighting the importance of evaluating their individual contributions and combined predictive value rather than relying solely on correlation strength.

#### Analyzing Monthly and Weekday Revenue Patterns

In [26]:
revenue_calendar_df = daily_revenue.rename("Daily_Revenue").reset_index()

revenue_calendar_df["Year_Month"] = (
    revenue_calendar_df["Transaction_Date"].dt.to_period("M").astype(str)
)

revenue_calendar_df["Day_of_Week"] = (
    revenue_calendar_df["Transaction_Date"].dt.day_name()
)

weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Sunday"
]

calendar_summary = (
    revenue_calendar_df.groupby(["Year_Month", "Day_of_Week"])
    .agg(
        Mean_Revenue=("Daily_Revenue", "mean"),
        Trading_Days=("Daily_Revenue", "count")
    )
    .reset_index()
)

revenue_heatmap = calendar_summary.pivot(
    index="Year_Month",
    columns="Day_of_Week",
    values="Mean_Revenue"
).reindex(columns=weekday_order)

trading_days_heatmap = calendar_summary.pivot(
    index="Year_Month",
    columns="Day_of_Week",
    values="Trading_Days"
).reindex(columns=weekday_order)

fig = go.Figure(
    data=go.Heatmap(
        z=revenue_heatmap.values,
        x=revenue_heatmap.columns,
        y=revenue_heatmap.index,
        colorscale="Blues",
        colorbar=dict(title="Average Revenue"),
        customdata=trading_days_heatmap.values,
        hovertemplate=(
            "Month: %{y}"
            "<br>Day: %{x}"
            "<br>Average Revenue: %{z:,.2f}"
            "<br>Trading Days: %{customdata:.0f}"
            "<extra></extra>"
        )
    )
)

fig.update_layout(
    title=dict(
        text="Average Daily Sales Revenue by Month and Day of the Week",
        x=0.5,
        xanchor="center"
    ),
    xaxis_title="Day of the Week",
    yaxis_title="Month",
    template="plotly_white",
    width=1100,
    height=650,
    font=dict(size=13)
)

fig.update_yaxes(autorange="reversed")

fig.show()

print("Average Daily Revenue by Month and Weekday:")
display(revenue_heatmap.round(2))

print("\nTrading Days by Month and Weekday:")
display(trading_days_heatmap.fillna(0).astype(int))

Average Daily Revenue by Month and Weekday:


Day_of_Week,Monday,Tuesday,Wednesday,Thursday,Friday,Sunday
Year_Month,,,,,,
2010-12,"36,945.84","55,535.96","33,753.32","38,557.58","46,040.18","18,531.17"
2011-01,"24,072.59","31,398.10","24,369.94","25,191.69","30,266.88","9,817.33"
2011-02,"26,546.27","29,775.68","21,446.34","20,811.08","19,554.61","7,010.26"
2011-03,"23,466.70","33,369.81","22,858.17","32,955.42","24,176.82","11,024.90"
2011-04,"25,802.27","26,213.06","23,600.62","26,228.00","24,201.35","9,605.09"
2011-05,"32,104.46","35,600.58","29,428.88","38,627.08","29,825.84","13,296.81"
2011-06,"20,903.32","33,416.81","29,715.53","35,444.03","29,113.64","16,395.79"
2011-07,"29,301.84","33,291.45","25,522.90","37,213.33","17,684.02","17,650.07"
2011-08,"22,088.27","25,191.96","33,243.02","50,165.00","20,823.71","9,548.20"



Trading Days by Month and Weekday:


Day_of_Week,Monday,Tuesday,Wednesday,Thursday,Friday,Sunday
Year_Month,,,,,,
2010-12,3,3,4,4,3,3
2011-01,4,4,4,4,4,4
2011-02,4,4,4,4,4,4
2011-03,4,5,5,5,4,4
2011-04,3,4,4,4,3,3
2011-05,3,5,4,4,4,5
2011-06,4,4,5,5,4,4
2011-07,4,4,4,4,5,5
2011-08,4,5,5,4,4,4


**Key Observations**
- **Sunday consistently records the lowest average daily revenue across all 13 observed months**, indicating a persistent weekly pattern despite changes in overall monthly sales performance.
- **Revenue generally strengthens across multiple weekdays during September–November 2011**, with November recording particularly high averages on Monday (73,644.42), Wednesday (59,562.44), and Thursday (58,682.78).
- **Most month–weekday combinations contain 3–5 trading days**, providing reasonably consistent observation counts. However, December 2011 contains only 1–2 observations per weekday, making its averages less representative of typical monthly patterns.

**Business Insights**
- **The persistent Sunday revenue gap and changing weekday performance across months suggest that both weekday and monthly calendar features may help capture variations in retail sales activity.**
- **Stronger revenue across several weekdays during September–November highlights the importance of preparing inventory, staffing, and fulfillment capacity for periods of increased retail activity**, while accounting for differences in trading-day availability.

#### Analyzing Retail Activity and Revenue Across Weekdays

In [27]:
weekday_metrics = daily_metrics.copy()

weekday_metrics["Day_of_Week"] = weekday_metrics.index.day_name()

weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Sunday"
]

metric_columns = [
    "Daily Revenue",
    "Transaction Volume",
    "Invoice Volume",
    "Net Merchandise Units"
]

weekday_averages = (
    weekday_metrics.groupby("Day_of_Week")[metric_columns]
    .mean()
    .reindex(weekday_order)
)

overall_averages = daily_metrics[metric_columns].mean()

weekday_index = weekday_averages.div(overall_averages).mul(100)

fig = px.bar(
    weekday_index.reset_index().melt(
        id_vars="Day_of_Week",
        var_name="Metric",
        value_name="Index"
    ),
    x="Day_of_Week",
    y="Index",
    color="Metric",
    barmode="group",
    title="Relative Retail Activity and Revenue Across Weekdays",
    labels={
        "Day_of_Week": "Day of the Week",
        "Index": "Index (Overall Daily Average = 100)"
    },
    template="plotly_white"
)

fig.add_hline(
    y=100,
    line_dash="dash",
    line_color="gray"
)

fig.update_traces(
    hovertemplate=(
        "Day: %{x}"
        "<br>Index: %{y:.1f}"
        "<extra></extra>"
    )
)

fig.update_layout(
    width=1100,
    height=550,
    title_x=0.5,
    xaxis_title="Day of the Week",
    yaxis_title="Index (Overall Daily Average = 100)",
    legend_title="Retail Metric",
    font=dict(size=13)
)

fig.show()

print("Weekday Activity Index (Overall Daily Average = 100):")
display(weekday_index.round(2))

print("\nAverage Daily Retail Metrics by Weekday:")
display(weekday_averages.round(2))

Weekday Activity Index (Overall Daily Average = 100):


,Daily Revenue,Transaction Volume,Invoice Volume,Net Merchandise Units
Day_of_Week,,,,
Monday,107.73,113.76,101.68,101.96
Tuesday,117.91,110.14,106.55,108.78
Wednesday,102.42,100.46,107.66,107.87
Thursday,122.49,110.29,126.09,130.08
Friday,98.02,92.43,98.62,93.61
Sunday,49.69,72.69,57.22,55.19



Average Daily Retail Metrics by Weekday:


,Daily Revenue,Transaction Volume,Invoice Volume,Net Merchandise Units
Day_of_Week,,,,
Monday,"34,589.99","2,010.38",84.34,"17,270.55"
Tuesday,"37,858.76","1,946.44",88.38,"18,426.37"
Wednesday,"32,887.11","1,775.28",89.30,"18,271.34"
Thursday,"39,329.51","1,949.02",104.58,"22,033.09"
Friday,"31,474.25","1,633.50",81.80,"15,855.98"
Sunday,"15,956.39","1,284.64",47.46,"9,348.48"


**Key Observations**
- **Thursday records the strongest overall retail activity**, with daily revenue (122.49), invoice volume (126.09), and net merchandise units (130.08) substantially exceeding their respective overall daily averages.
- **Tuesday generates above-average performance across all four metrics**, while Monday records the highest transaction volume index (113.76), indicating that the busiest days by invoice-line count do not necessarily generate the highest revenue.
- **Sunday consistently records the lowest activity across all metrics**, with revenue falling to 49.69% of the overall daily average, compared with 72.69% for transaction volume, 57.22% for invoice volume, and 55.19% for net merchandise units.

**Business Insights**
- **Thursday's strong performance across revenue, invoice volume, and merchandise quantities highlights the importance of aligning inventory availability, staffing, and fulfillment capacity with recurring weekday activity patterns.**
- **Differences between revenue and transaction-volume indices suggest that sales performance depends on more than invoice-line counts alone**, reinforcing the value of examining merchandise quantities, invoice activity, and calendar patterns when developing forecasting features.